## this script cleans the gender information of authors. for each cluster_id, choose the most frequent gender category.

### input
- author_info: the data is derived from [BDVincent].[dbo].[gender_prod_80]

### output
- pub_author: the file maps paper id with author id
- author_info: the file contains demographic information of authors

In [1]:
import pandas as pd
import collections

In [2]:
info_path='../../data/dropbox/LaborDivision/data/rawdata/article_author_info.txt'

In [43]:
pub_author_path='../../data/dropbox/LaborDivision/Data/cleanedraw/pub_authorid.pkl'
info_new_path='../../data/dropbox/LaborDivision/Data/cleanedraw/author_info.pkl'

In [7]:
info_df= pd.read_csv(info_path,sep=",")
info_df.head()

/tmp/ipykernel_909640/3360861756.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  info_df= pd.read_csv(info_path,sep=",")


,OST_BK,Seq_No,cluster_id,gender,YFP,npapa
0,107154401,5.0,81178534.0,F,2012.0,11.0
1,108260661,745.0,4745329.0,F,2013.0,746.0
2,105432460,9.0,30161693.0,U,2013.0,1.0
3,40628747,1.0,88125653.0,M,2003.0,111.0
4,50934034,6.0,22288059.0,M,2000.0,258.0


In [19]:
info_df.sort_values('cluster_id')

,cluster_id,gender,YFP,npapa
97048,4.0,M,1979.0,113.0
9234824,5.0,F,2010.0,3.0
54024859,7.0,F,1996.0,9.0
463467,8.0,F,2000.0,40.0
68526319,9.0,F,2012.0,6.0
...,...,...,...,...
38395384,94809624.0,U,2020.0,3.0
34732020,94809625.0,U,2018.0,1.0
4127368,94809632.0,F,2005.0,15.0
101901254,94809633.0,U,2018.0,1.0


In [8]:
info_df = info_df[['cluster_id','gender','YFP','npapa']].drop_duplicates()

In [9]:
info_df['gender'] = info_df['gender'].fillna('U')
info_df = info_df.replace(to_replace={'gender':{'m':'M','f':'F'}})

In [14]:
info_df['gender'].unique()

array(['F', 'U', 'M'], dtype=object)

## extract the paper_author mapping relation

In [14]:
pub_author=info_df[['id','seqno','cluster_id']]
pub_author.to_pickle(pub_author_path)

## assign authors to its most frequent gender

In [31]:
gender_df=info_df[['cluster_id','first_name','gender']].drop_duplicates()

In [32]:
# standardize
gender_df=gender_df.replace(to_replace={'gender':{'m':'M','f':'F','INI':'unk','UNI':'unk','UNK':'unk'}})
gender_df=gender_df.fillna(value='unk')

In [33]:
gender_df = gender_df.groupby(['cluster_id','gender']).size().reset_index(name='count')
gender_df=gender_df.pivot_table(index='cluster_id',columns='gender',values='count')
gender_df=gender_df.fillna(0)
id_gender=gender_df.idxmax(axis=1).reset_index(name='gender')#return the column has largest value

In [21]:
info_df.head()

,id,seqno,first_name,cluster_id,gender,yfp,npub
0,75871588,2,S. H.,689940,INI,2008,1
1,53614014,3,U.,44823147,INI,2008,1
2,78030267,1,Rui,27676457,M,2011,1
3,54383423,2,Madeleine,59751143,F,2018,1
4,32237142,3,Frank,59894652,M,2004,8


In [34]:
id_gender.gender.unique()

array(['M', 'F', 'unk'], dtype=object)

In [35]:
info_new=info_df[['cluster_id','yfp','npub']]
info_new=info_new.drop_duplicates(subset=['cluster_id'])

In [36]:
info_new=info_new.merge(id_gender,on='cluster_id')

In [37]:
## count the gender identification distribution of authors
collections.Counter(info_new['gender'].values.tolist())

Counter({'unk': 5610284, 'M': 12550338, 'F': 7275226})

In [38]:
5610284/(5610284+12550338+7275226)

0.22056602948720247

In [40]:
info_new.head()

,cluster_id,yfp,npub,gender
0,689940,2008,1,unk
1,44823147,2008,1,unk
2,27676457,2011,1,M
3,59751143,2018,1,F
4,59894652,2004,8,M


In [41]:
info_new=info_new.astype({'yfp':'int16','npub':'int16','cluster_id':'str'})

In [44]:
info_new.to_pickle(info_new_path)